# Data Preprocessing

## 1. Introduction

This notebook demonstrates the data preparation and preprocessing workflow for the IBM Telco Customer Churn dataset.

The workflow covers feature and target separation, stratified data splitting, deterministic data cleaning, and feature preprocessing.

The preprocessing logic is reused from the project's source modules to maintain consistency between notebook experiments and the final machine learning pipeline.

The training split is used to fit the preprocessing transformations, while the validation and test splits are only transformed using the fitted preprocessing pipeline.

## 2. Loading the Dataset

In [1]:
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from churn_prediction.data import load_data, split_features_target

df = load_data()

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## 3. Separating Features and Target

In [2]:
X, y = split_features_target(df)

print("Features shape:", X.shape)
print("Target shape:", y.shape)

print("\nTarget distribution:")
print(y.value_counts())

print("\nTarget proportions:")
print(y.value_counts(normalize=True).round(4))

Features shape: (7043, 20)
Target shape: (7043,)

Target distribution:
Churn
0    5174
1    1869
Name: count, dtype: int64

Target proportions:
Churn
0    0.7346
1    0.2654
Name: proportion, dtype: float64


## 4. Train, Validation and Test Split

In [3]:
from churn_prediction.train import split_data

X_fit, X_val, X_test, y_fit, y_val, y_test = split_data(X, y)

print("Training samples:", len(X_fit))
print("Validation samples:", len(X_val))
print("Test samples:", len(X_test))

Training samples: 4507
Validation samples: 1127
Test samples: 1409


In [4]:
split_summary = pd.DataFrame({
    "Training": y_fit.value_counts(normalize=True),
    "Validation": y_val.value_counts(normalize=True),
    "Test": y_test.value_counts(normalize=True),
}).rename(index={0: "No Churn", 1: "Churn"})

split_summary.round(4)

,Training,Validation,Test
Churn,,,
No Churn,0.7346,0.7347,0.7346
Churn,0.2654,0.2653,0.2654


## 5. Deterministic Data Cleaning

The raw input features require a small number of deterministic cleaning steps before preprocessing.

The `clean_features` function from the project's source module removes the customer identifier and converts `TotalCharges` to a numeric feature. Missing values in `TotalCharges` are replaced with zero, consistent with the dataset's initial analysis.

These operations do not require fitting and can be applied independently to each data split.

In [5]:
from churn_prediction.preprocessing import clean_features

print("Original training features:")
print("Shape:", X_fit.shape)
print("Columns:", X_fit.columns.tolist())

print("\nMissing TotalCharges values:")
print(X_fit["TotalCharges"].astype(str).str.strip().eq("").sum())

Original training features:
Shape: (4507, 20)
Columns: ['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges']

Missing TotalCharges values:
7


In [6]:
X_fit_clean = clean_features(X_fit)
X_val_clean = clean_features(X_val)
X_test_clean = clean_features(X_test)

print("Training features after cleaning:", X_fit_clean.shape)
print("Validation features after cleaning:", X_val_clean.shape)
print("Test features after cleaning:", X_test_clean.shape)

print("\nRemaining missing values in TotalCharges:")
print(X_fit_clean["TotalCharges"].isna().sum())

print("\nCustomerID removed:", "customerID" not in X_fit_clean.columns)

Training features after cleaning: (4507, 19)
Validation features after cleaning: (1127, 19)
Test features after cleaning: (1409, 19)

Remaining missing values in TotalCharges:
0

CustomerID removed: True


In [7]:
print(X_fit_clean[["tenure", "MonthlyCharges", "TotalCharges"]].dtypes)

X_fit_clean[["tenure", "MonthlyCharges", "TotalCharges"]].describe().round(2)

tenure              int64
MonthlyCharges    float64
TotalCharges      float64
dtype: object


,tenure,MonthlyCharges,TotalCharges
count,4507.00,4507.00,4507.00
mean,32.30,64.70,2283.09
std,24.66,30.02,2279.04
min,0.00,18.40,0.00
25%,9.00,35.52,383.60
50%,29.00,70.35,1380.40
75%,56.00,89.75,3812.98
max,72.00,118.75,8684.80


## 6. Defining the Preprocessing Pipeline

After deterministic cleaning, numerical and categorical features require different preprocessing strategies.

The project's `build_preprocessor` function defines the transformations used by the final Gradient Boosting model:

- Numerical features are passed through without scaling.
- Missing categorical values are imputed using the most frequent value.
- Categorical features are encoded using One-Hot Encoding.
- Previously unseen categories are handled using `handle_unknown="ignore"`.

The preprocessing configuration is reused from the project's source module to ensure consistency with the final model pipeline.

In [8]:
from churn_prediction.preprocessing import (
    build_preprocessor,
    NUMERICAL_FEATURES,
    CATEGORICAL_FEATURES,
)

preprocessor = build_preprocessor()

print("Numerical features:", NUMERICAL_FEATURES)
print("\nCategorical features:", CATEGORICAL_FEATURES)

Numerical features: ['tenure', 'MonthlyCharges', 'TotalCharges']

Categorical features: ['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']


## 7. Fitting and Applying Preprocessing

The preprocessing pipeline is fitted exclusively on the training data.

The fitted transformations are then applied to the validation and test sets without refitting. This prevents information from these evaluation sets from influencing the learned preprocessing parameters.

In [9]:
X_fit_processed = preprocessor.fit_transform(X_fit_clean)

X_val_processed = preprocessor.transform(X_val_clean)

X_test_processed = preprocessor.transform(X_test_clean)

print("Training processed shape:", X_fit_processed.shape)
print("Validation processed shape:", X_val_processed.shape)
print("Test processed shape:", X_test_processed.shape)

Training processed shape: (4507, 46)
Validation processed shape: (1127, 46)
Test processed shape: (1409, 46)


## 8. Preprocessing Verification

The transformed datasets are checked to verify that preprocessing produces a consistent feature space across all data splits.

The output feature names are also inspected to confirm that numerical features remain unchanged and categorical features are represented through One-Hot Encoding.

In [10]:
print("Number of numerical features:", len(NUMERICAL_FEATURES))
print("Number of categorical features:", len(CATEGORICAL_FEATURES))

print("\nNumber of processed features:", X_fit_processed.shape[1])

assert X_fit_processed.shape[1] == X_val_processed.shape[1]
assert X_fit_processed.shape[1] == X_test_processed.shape[1]

print("\nAll splits have the same number of processed features.")

Number of numerical features: 3
Number of categorical features: 16

Number of processed features: 46

All splits have the same number of processed features.


In [11]:
feature_names = preprocessor.get_feature_names_out()

print("Total feature names:", len(feature_names))
print("\nFirst 20 processed features:")

print(feature_names[:20])

Total feature names: 46

First 20 processed features:
['num__tenure' 'num__MonthlyCharges' 'num__TotalCharges'
 'cat__gender_Female' 'cat__gender_Male' 'cat__SeniorCitizen_0'
 'cat__SeniorCitizen_1' 'cat__Partner_No' 'cat__Partner_Yes'
 'cat__Dependents_No' 'cat__Dependents_Yes' 'cat__PhoneService_No'
 'cat__PhoneService_Yes' 'cat__MultipleLines_No'
 'cat__MultipleLines_No phone service' 'cat__MultipleLines_Yes'
 'cat__InternetService_DSL' 'cat__InternetService_Fiber optic'
 'cat__InternetService_No' 'cat__OnlineSecurity_No']


In [12]:
import numpy as np
from scipy import sparse

def check_processed_data(data, name):
    if sparse.issparse(data):
        has_missing = np.isnan(data.data).any()
    else:
        has_missing = np.isnan(data).any()

    print(f"{name}:")
    print("  Shape:", data.shape)
    print("  Contains missing values:", has_missing)


check_processed_data(X_fit_processed, "Training")
check_processed_data(X_val_processed, "Validation")
check_processed_data(X_test_processed, "Test")

Training:
  Shape: (4507, 46)
  Contains missing values: False
Validation:
  Shape: (1127, 46)
  Contains missing values: False
Test:
  Shape: (1409, 46)
  Contains missing values: False


## 9. Summary and Modeling Implications

This notebook demonstrated the data preparation and preprocessing workflow for the customer churn prediction project.

The main steps included:

- Loading the raw dataset and separating features from the target.
- Splitting the data into training, validation, and test sets using stratified sampling.
- Removing the customer identifier and converting `TotalCharges` to a numeric feature.
- Applying the project's preprocessing pipeline to numerical and categorical features.
- Fitting preprocessing transformations exclusively on the training set and applying them to the validation and test sets.

The final preprocessing pipeline produces 46 features, including 3 numerical features and 43 one-hot encoded categorical features.

The preprocessing logic is consistent with the pipeline used by the final Gradient Boosting model.

The resulting datasets are ready for model training and evaluation. However, the test set was consulted during earlier exploratory model comparisons, so its results should be interpreted as descriptive rather than as an independent final estimate of model performance.